Import Libraries

In [2]:
import paramiko #Library to communicate to raspberry pi and execute commands
import re #library for regular expression
import serial #library to serialize the communication between pc and printer
import time #library for time and sleep
import tensorflow as tf #library to load and run model
from tensorflow.keras.preprocessing import image #library to preprocess the image
import numpy as np #library to load the image into an array

Initialize Variables

In [3]:
port = 'COM8' #port fo r 3 D printer
baudrate = 115200 #baudrate of printer
host = '10.203.13.39' #ip address of the printer
username = 'raspi' #username for pi
password = 'mero2023' #password for the pi
file = 'optimized_circle_20layer.gcode' #gcode file location
capture_interval = 10 #delay for capturing image
destination = 'C:\\Users\\anubh\\Pictures\\Pi\\' #store location for the image

Function to execute a command

In [4]:
def execute_command(port, baudrate,command):
    if command.startswith(";"): #skip the comments
        return
    with serial.Serial(port, baudrate) as ser: #open the serial port
        ser.write(command.encode())  # Send the command as a byte string
        response = ser.readline().decode().strip() #decode the response from printer
        if response != 0: #if no error is recieved
            while 'ok' not in response: #wait till the printer has successfully executed command
                response = ser.readline().decode().strip()
                continue
            return response #return the response
        else:
            print("Failed to parse the response.") #Else print error

Function to take picture

In [5]:
def take_picture(destination_folder, ssh_client, sftp_client): #take the pictures destination , ssh and sftp clients
    timestamp = time.strftime('%Y%m%d%H%M%S') #generate tme stamp
    filename = f'image_{timestamp}.jpeg' #generate unique image name using timestamp

    ssh_client.get_transport() #get the transport layer
    comm = "sudo -S fuser /dev/video0" #command to check if any other process is using camera
    stdin, stdout, stderr = ssh_client.exec_command(command=comm,get_pty=True) #send the command
    stdin.write("mero2023\n") #send the password
    stdin.flush()

    if stderr.channel.recv_exit_status() != 0: #if error is recieved
        print(f"{stderr.readlines()}") #print the error
    else:
        pid =int(re.search(r'\s(\d+)m', stdout.readlines()[2]).group(1)) #get the process id which is using the camera
        if pid!= None:
            com = f"sudo -S kill {pid}" #if process id is found create command to kill the process
            stdin, stdout, stderr = ssh_client.exec_command(command=com,get_pty=True) #execute the command on ssh
            stdin.write("mero2023\n") #send the password
            stdin.flush()
    time.sleep(2)  #sleep for 2 seconds
    capture_command = f'libcamera-jpeg -o {filename}' #command to capture image
    ssh_client.exec_command(capture_command) #execute the command

    # Wait for the specified interval
    time.sleep(capture_interval)

    # Transfer the captured image to your laptop
    remote_path = f'{filename}' #remote path to image
    local_path = f'{destination_folder}/{filename}' #local path to image
    sftp_client.get(remote_path, local_path) #get the image using sftp client
    return local_path #return the path to image

Function to connect to pi using ssh

In [6]:
def connect(ssh_host,ssh_username, ssh_password):
    # Establish SSH connection
    ssh_client = paramiko.SSHClient() #get the ssh client
    ssh_client.set_missing_host_key_policy(paramiko.AutoAddPolicy()) #set policy
    ssh_client.connect(ssh_host, username=ssh_username, password=ssh_password) #connect to ssh 
    # Create SFTP client for file transfer
    sftp_client = ssh_client.open_sftp()
    return ssh_client, sftp_client #return the clients

In [7]:
ssh, sftp = connect(host, username, password) #get the clients to ssh and sftp

In [8]:
# Load the saved model
model = tf.keras.models.load_model('crop_model.h5')

Function to run image through model

In [9]:
def run_model(img_path, mod):
    message = "Layer is good!"
    # Load and preprocess the image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0

    # Predict the class probabilities
    predictions = mod.predict(img_array)
    predicted_class_index = np.argmax(predictions[0])

    # Get the confidence score of the predicted class
    confidence = predictions[0][predicted_class_index] * 100
    # Get the predicted class label
    predicted_class = np.argmax(predictions)
    if predicted_class == 0:
        message = "Good\n"
    if predicted_class == 1:
        message = "High Level \n"
    if predicted_class == 2:
        message = "Flowrate High\n"
    if predicted_class == 3:
        message = "Low Bed Level\n"
    if predicted_class == 4:
        message = "Low Flow\n"
    # Print the predicted class label
    return message+" : "+str(confidence)

In [10]:
import tkinter as tk
from tkinter import messagebox

def display_message_box(message):
    def on_ok():
        nonlocal flag
        flag = True
        window.destroy()

    def on_cancel():
        nonlocal flag
        flag = False
        window.destroy()

    # Create the main window
    window = tk.Tk()
    window.title("GUI Example")
    window.geometry("300x150")  # Set the window size to be more square

    # Create the message box
    message_label = tk.Label(window, text=message)
    message_label.pack(pady=20)

    # Create the OK button
    ok_button = tk.Button(window, text="OK", command=on_ok)
    ok_button.pack(side=tk.LEFT, padx=20)

    # Create the Cancel button
    cancel_button = tk.Button(window, text="Cancel", command=on_cancel)
    cancel_button.pack(side=tk.RIGHT, padx=20)

    # Initialize the flag
    flag = None

    # Start the main event loop
    window.mainloop()

    return flag


In [11]:
with open(file, "r") as f: #open the file
    for line in f: #loop through to get g-code commads
        if line is not None: #is code is not null
            execute_command(port, baudrate, line+"\n") #execute the command
            if "M240" in line: #if snap photo command is encountered
                image_path = take_picture(destination, ssh, sftp) #take a picture on pi and trnasfer to local memory
                prediction = run_model(image_path, mod=model) #run it through model to get the prediction
                if display_message_box(prediction) != True: #if the dialog box is dismissed break out of loop
                    break

execute_command(port, baudrate, "G28") #Home the printer

1/1 [==============================] - 0s 59ms/step
[]
1/1 [==============================] - 0s 49ms/step
[]
1/1 [==============================] - 0s 55ms/step


'ok'